In [2]:
import os
import numpy as np
import rasterio
from collections import defaultdict
from tqdm import tqdm

input_dir = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Code_Alba/predictions'
output_file = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Code_Alba/final_yearly_predictions.tif'

# Create a dict to hold lists of predictions for each year
yearly_preds = defaultdict(list)

# Collect metadata from the first file
first_file = sorted(os.listdir(input_dir))[0]
with rasterio.open(os.path.join(input_dir, first_file)) as src:
    meta = src.meta.copy()
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs
    dtype = src.dtypes[0]

# Traverse all files with error handling
for fname in tqdm(sorted(os.listdir(input_dir))):
    if not fname.endswith('.tif'):
        continue
    path = os.path.join(input_dir, fname)
    
    try:
        with rasterio.open(path) as src:
            # Read in blocks to handle large files
            block_shape = (256, 256)
            
            for i in range(1, src.count + 1):
                try:
                    # Read band data in blocks
                    band_data = np.zeros((height, width), dtype=np.uint8)
                    
                    # Process each block
                    for ji, window in src.block_windows(i):
                        block_data = src.read(i, window=window)
                        band_data[window.row_off:window.row_off + window.height,
                                window.col_off:window.col_off + window.width] = block_data
                    
                    # Get year from band description
                    band_year = int(src.descriptions[i - 1].split()[-1])  # "Predictions for year 1991"
                    
                    # Verify data is valid
                    if not np.isfinite(band_data).all():
                        print(f"Warning: Found non-finite values in {fname}, band {i}, replacing with 0")
                        band_data = np.nan_to_num(band_data, nan=0, posinf=0, neginf=0)
                    
                    yearly_preds[band_year].append(band_data)
                    
                except Exception as e:
                    print(f"Error reading band {i} from {fname}: {e}")
                    # Create dummy data for this band
                    dummy_data = np.zeros((height, width), dtype=np.uint8)
                    yearly_preds[band_year].append(dummy_data)
                    
    except Exception as e:
        print(f"Error opening file {fname}: {e}")
        continue

# Final sorted list of years
sorted_years = sorted(yearly_preds.keys())

# Aggregate predictions with more sensitive detection
aggregated_predictions = []
for year in sorted_years:
    stack = np.stack(yearly_preds[year], axis=0)  # Shape: [n_predictions, H, W]
    
    # Count how many times each pixel is predicted as disturbed
    disturbance_count = np.sum(stack > 0, axis=0)
    
    # If any prediction shows disturbance, mark it (more sensitive)
    vote = (disturbance_count > 0).astype(np.uint8)
    
    # Print statistics for this year
    total_pixels = vote.size
    disturbed_pixels = np.sum(vote)
    print(f"\nYear {year}:")
    print(f"Number of predictions for this year: {len(yearly_preds[year])}")
    print(f"Pixels marked as disturbed: {disturbed_pixels} ({(disturbed_pixels/total_pixels)*100:.2f}%)")
    
    aggregated_predictions.append(vote)

# Save final raster with simple, robust settings
meta.update({
    "count": len(sorted_years),
    "dtype": rasterio.uint8,
    "compress": "lzw",  # Simple, widely supported compression
    "driver": "GTiff",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
    "interleave": "band",  # Required setting
    "bigtiff": "YES",  # Support for large files
    "predictor": 2,  # Horizontal differencing for better compression
    "zlevel": 1  # Light compression for better compatibility
})

try:
    # First verify all arrays are valid
    for pred in aggregated_predictions:
        if not np.isfinite(pred).all():
            print("Warning: Found non-finite values in predictions, replacing with 0")
            pred[~np.isfinite(pred)] = 0
        if pred.dtype != np.uint8:
            print(f"Warning: Converting array from {pred.dtype} to uint8")
            pred = pred.astype(np.uint8)

    # Write with error handling
    with rasterio.open(output_file, "w", **meta) as dst:
        for i, year in enumerate(sorted_years):
            try:
                # Write band
                dst.write(aggregated_predictions[i], i + 1)
                dst.set_band_description(i + 1, f"Final prediction for year {year}")
            except Exception as e:
                print(f"Error writing band {i+1} (year {year}): {e}")
                raise

    # Verify the file was written correctly
    with rasterio.open(output_file) as src:
        print("\nVerifying output file:")
        print(f"Number of bands: {src.count}")
        print(f"File size: {os.path.getsize(output_file) / (1024*1024):.1f} MB")
        
        # Read a small test block from each band
        for i in range(src.count):
            test_data = src.read(i + 1, window=((0, 256), (0, 256)))
            print(f"Band {i+1} test block shape: {test_data.shape}, dtype: {test_data.dtype}")

    print(f"\nSuccessfully saved and verified final aggregated predictions to {output_file}")

except Exception as e:
    print(f"\nError saving predictions: {e}")
    print("Trying alternative save method...")
    
    # Try alternative save method with minimal settings
    basic_meta = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": len(sorted_years),
        "dtype": rasterio.uint8,
        "crs": crs,
        "transform": transform,
        "interleave": "band",  # Required setting
        "bigtiff": "YES",  # Support for large files
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256
    }
    
    with rasterio.open(output_file, "w", **basic_meta) as dst:
        for i, year in enumerate(sorted_years):
            dst.write(aggregated_predictions[i], i + 1)
            dst.set_band_description(i + 1, f"Final prediction for year {year}")
    
    print(f"Saved with basic settings to {output_file}")

# Final verification
try:
    with rasterio.open(output_file) as src:
        print("\nFinal verification:")
        print(f"File exists: {os.path.exists(output_file)}")
        print(f"File size: {os.path.getsize(output_file) / (1024*1024):.1f} MB")
        print(f"Number of bands: {src.count}")
        print(f"Data type: {src.dtypes[0]}")
        print(f"CRS: {src.crs}")
        print(f"Transform: {src.transform}")
        
        # Test read a small block from first and last band
        first_block = src.read(1, window=((0, 256), (0, 256)))
        last_block = src.read(src.count, window=((0, 256), (0, 256)))
        print(f"\nFirst band test block shape: {first_block.shape}")
        print(f"Last band test block shape: {last_block.shape}")
except Exception as e:
    print(f"\nError in final verification: {e}")


100%|██████████| 33/33 [01:11<00:00,  2.18s/it]



Year 1984:
Number of predictions for this year: 1
Pixels marked as disturbed: 2646159 (10.58%)

Year 1985:
Number of predictions for this year: 2
Pixels marked as disturbed: 1958388 (7.83%)

Year 1986:
Number of predictions for this year: 3
Pixels marked as disturbed: 3624963 (14.50%)

Year 1987:
Number of predictions for this year: 4
Pixels marked as disturbed: 3276381 (13.11%)

Year 1988:
Number of predictions for this year: 5
Pixels marked as disturbed: 2334531 (9.34%)

Year 1989:
Number of predictions for this year: 6
Pixels marked as disturbed: 5279703 (21.12%)

Year 1990:
Number of predictions for this year: 7
Pixels marked as disturbed: 5188147 (20.75%)

Year 1991:
Number of predictions for this year: 8
Pixels marked as disturbed: 3839932 (15.36%)

Year 1992:
Number of predictions for this year: 8
Pixels marked as disturbed: 5234634 (20.94%)

Year 1993:
Number of predictions for this year: 8
Pixels marked as disturbed: 3234729 (12.94%)

Year 1994:
Number of predictions for this